# Website Chatter RAG

You provide a company website link, and you can talk to an LLM about it. Useful for technical people to interact with complex api documentation.

1. User provides a website URL
2. Scrape all available information (Beautiful Soup + Playwright)
3. Chunk the content (LangChain)
4. Vectorize chunks (LangChain + OpenAI)
5. Store in Chroma (LangChain)
6. Setup retriever
7. Gradio interface to chat with the website using its content as context

All RAG uses LangChain.

In [1]:
# Install Playwright browser if needed (run once)
!playwright install chromium

In [4]:
import gradio as gr
from rag_pipeline import ingest, answer

# Module-level vectorstore updated when user loads a website
_current_vectorstore = None


def load_website(url: str):
    """Scrape URL, chunk, embed, store in Chroma. Returns status."""
    global _current_vectorstore
    if not url or not url.strip():
        return "Please enter a valid URL."
    url = url.strip()
    if not url.startswith(("http://", "https://")):
        url = "https://" + url
    try:
        _current_vectorstore = ingest.scrape_and_ingest(url)
        return f"Loaded: {url}"
    except Exception as e:
        _current_vectorstore = None
        return f"Error: {e}"


def chat_fn(message, history):
    """RAG chat handler. Uses _current_vectorstore from load_website."""
    if _current_vectorstore is None:
        return "Please load a website first (enter URL and click 'Load Website')."
    # Convert Gradio history to [{"role": ..., "content": ...}, ...]
    hist = []
    for item in (history or []):
        if isinstance(item, (list, tuple)) and len(item) >= 2:
            hist.append({"role": "user", "content": str(item[0]) or ""})
            hist.append({"role": "assistant", "content": str(item[1]) or ""})
        elif isinstance(item, dict) and "role" in item and "content" in item:
            hist.append({"role": item["role"], "content": str(item.get("content", ""))})
    response, _ = answer.answer_question(message, hist, vectorstore=_current_vectorstore)
    return response

In [ ]:
with gr.Blocks(title="Website Chatter RAG") as ui:
    gr.Markdown("# Website Chatter RAG")
    gr.Markdown("Enter a URL, load the website, then ask questions about its content.")

    with gr.Row():
        url_input = gr.Textbox(
            label="Website URL",
            placeholder="https://example.com",
            scale=4,
        )
        load_btn = gr.Button("Load Website", variant="primary", scale=1)
    status = gr.Markdown("Enter a URL and click 'Load Website' to start.")

    load_btn.click(
        fn=load_website,
        inputs=[url_input],
        outputs=[status],
    )

    gr.ChatInterface(
        fn=chat_fn,
        type="messages",
        title="Chat with the website",
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Home - Edward Donner

Home
AI Curriculum
Proficient AI Engineer
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 400,000 enrolled across 190